In [3]:
import numpy as np

In [4]:
# Generate a sequence with some noise
def generate_sequence(length, noise=0.1):
    return np.array([i + np.random.normal(0, noise) for i in range(length)])

In [5]:
# Convert sequence into input/output pairs
def create_dataset(seq, n_stps):
    X, y = [], []
    for i in range(len(seq) - n_stps):
        X.append(seq[i:i + n_stps])
        y.append(seq[i + n_stps])
    return np.array(X), np.array(y)

In [6]:
# Prepare data
np.random.seed(42)
sequence = generate_sequence(100, noise=0.5)

In [7]:
n_steps = 3
X, y = create_dataset(sequence, n_steps)
X = X.reshape((X.shape[0], X.shape[1], 1))

In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import SimpleRNN, Dense

In [9]:
# Build model WITH regularization and gradient clipping
model = Sequential([
    SimpleRNN(
        50,
        activation='relu',
        input_shape=(n_steps, 1),
        kernel_regularizer=l2(0.01),
        recurrent_regularizer=l2(0.01)
    ),
    Dense(1)
])

C:\Users\cfl602\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
# Compile with gradient-clipped optimizer
optimizer = tf.keras.optimizers.Adam(clipvalue=1.0)
model.compile(optimizer=optimizer, loss="mse")

In [11]:
# Train
model.fit(X, y, epochs=100, verbose=0)

In [12]:
# Predict
x_input = np.array([97, 98, 99])
x_input = x_input.reshape((1, n_steps, 1))
yhat = model.predict(x_input, verbose=0)
print(f"Predicted next value: {yhat[0][0]:.4f}")
print(f"Actual next value: ~100")

Predicted next value: 99.2398
Actual next value: ~100


In [13]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
import matplotlib.pyplot as plt

In [15]:
def generate_temperature_data(seq_length, num_samples):
    np.random.seed(42)
    base_temp = 20      # Average temperature in degrees Celsius
    temp_variation = 5  # Maximum daily temperature variation

    data = base_temp + temp_variation * np.sin(np.linspace(0, 100, num_samples)) + np.random.normal(0, 1, num_samples)

    X = []
    y = []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])

    return np.array(X), np.array(y)


seq_length = 30
num_samples = 365  # Simulating a year of data
X, y = generate_temperature_data(seq_length, num_samples)
X = X[..., np.newaxis]  # Add an extra dimension for compatibility with RNN input

In [16]:
from sklearn.model_selection import train_test_split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 106ms/step - loss: 372.2900 - val_loss: 366.8335
Epoch 2/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 331.2557 - val_loss: 328.7849
Epoch 3/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 297.3134 - val_loss: 298.2711
Epoch 4/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - loss: 267.5370 - val_loss: 266.9782
Epoch 5/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 234.5494 - val_loss: 232.0365
Epoch 6/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 201.9543 - val_loss: 201.5849
Epoch 7/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 173.1283 - val_loss: 172.6340
Epoch 8/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 146.4765 - val_loss: 145.9796
Epoch 9/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 123.5714 - val_loss: 125.7590
Epoch 10/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 107.6940 - val_loss: 111.3501
Epoch 11/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 96.0956 - val_loss: 101.2526
Epoch 12/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 1

In [19]:
def predict_future(model, recent_data, days):
    predictions = []
    input_seq = recent_data[-seq_length:]

    for _ in range(days):
        pred = model.predict(input_seq[np.newaxis, ...])[0, 0]
        predictions.append(pred)
        input_seq = np.append(input_seq[1:], [[pred]], axis=0)

    return predictions


In [20]:
future_predictions = predict_future(model, X[-1], 7)
print("Predicted Temperatures for Next 7 Days: ", future_predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 236ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Predicted Temperatures for Next 7 Days:  [np.float32(13.886723), np.float32(13.871241), np.float32(13.870124), np.float32(13.870126), np.float32(13.870123), np.float32(13.870124), np.float32(13.870125)]
